In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense
from keras.models import Sequential

In [4]:
from pathlib import Path
import sys

dataset_root = Path(
    "/content/drive/MyDrive/RM_Thyroid/TN5000_forReview/TN5000_forReview"
)

image_dir = dataset_root / "JPEGImages"
annotation_dir = dataset_root / "Annotations"

# YOLO directory
yolo_dir = dataset_root / "YOLO_for_detection"

# YOLO labels output directory
yolo_labels_dir = yolo_dir / "yolo_labels"

img_files = sorted(image_dir.glob("*.jpg"))

print(f"Total images: {len(img_files)}")

Total images: 5000


#Checking a img file

In [5]:
import cv2
from google.colab.patches import cv2_imshow


img = cv2.imread(str(img_files[10]))
img.shape

print(f"Total images: {len(img_files)}")

Total images: 5000


In [6]:
print("Height:", img.shape[0])
print("Width :", img.shape[1])
print("Channels:", img.shape[2])
print("Data type:", img.dtype)
print("File size (KB):", round(image_dir.stat().st_size / 1024, 2))

Height: 628
Width : 818
Channels: 3
Data type: uint8
File size (KB): 4.0


In [ ]:
from PIL import Image
import xml.etree.ElementTree as ET
from collections import Counter

sizes = []
counter = Counter()

for img_path in img_files:
    xml_path = annotation_dir / f"{img_path.stem}.xml"

    # Store image size
    with Image.open(img_path) as img:
        sizes.append(img.size)

    # Read corresponding annotation
    if xml_path.exists():
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Count every <object>'s <name>
        for obj in root.findall("object"):
            name = obj.find("name").text.strip()
            counter[name] += 1

# Print counts
print("Class distribution:")
for cls in sorted(counter.keys()):
    print(f"Class {cls}: {counter[cls]}")

size_counts = Counter(sizes)
for size, count in size_counts.items():
  print(f"{size}: {count} images")

# Print image size information
print("\nNumber of images:", len(sizes))
print("Unique image sizes:", set(sizes))
print("Some sizes", list(set(sizes))[:10])

In [ ]:
import sys
import importlib

sys.path.append(str(yolo_dir))

from xml_to_yolo import XML_to_YOLO
converter = XML_to_YOLO(
    annotation_dir,
    yolo_labels_dir
)

converter.convert_all()

In [ ]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split


train_imgs, temp_imgs = train_test_split(
    img_files,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

for split in ["train", "val", "test"]:
    (yolo_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

def copy_files(image_list, split):

    for img_path in image_list:

        shutil.copy2(
            img_path,
            yolo_dir / "images" / split / img_path.name
        )

        txt_path = yolo_labels_dir / f"{img_path.stem}.txt"

        if txt_path.exists():

            shutil.copy2(
                txt_path,
                yolo_dir / "labels" / split / txt_path.name
            )

        else:
            print(f"Missing label: {txt_path.name}")

copy_files(train_imgs, "train")
copy_files(val_imgs, "val")
copy_files(test_imgs, "test")

print(f"Train: {len(train_imgs)}")
print(f"Validation: {len(val_imgs)}")
print(f"Test: {len(test_imgs)}")

print("YOLO dataset created successfully!")

In [ ]:
from pathlib import Path

yaml_text = f"""path: {yolo_dir}

train: images/train
val: images/val
test: images/test

names:
  0: nodule
"""

with open(yolo_dir / "data.yaml", "w") as f:
    f.write(yaml_text)

print("data.yaml created.")

In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics
ultralytics.checks()

# Classification using CNN + ViT


In [ ]:
import sys
import importlib
from ROIDataset import ROIDataset


sys.path.insert(0, str(roi_root))

train_path = yolo_dir / "images" / "train"
test_path = yolo_dir / "images" / "test"
val_path = yolo_dir / "images" / "val"


train_ds = ROIDataset(annotation_dir, train_path)
test_ds = ROIDataset(annotation_dir, test_path)
val_ds = ROIDataset(annotation_dir, val_path)

print(dir(train_ds))
print(dir(test_ds))
print(dir(val_ds))

In [ ]:
from pathlib import Path

roi_root = dataset_root / "ROI_Dataset" / "ROI_Images"

train_ds.save_all_roi(roi_root / "train")
val_ds.save_all_roi(roi_root / "val")
test_ds.save_all_roi(roi_root / "test")

In [ ]:
from pathlib import Path
import cv2
from google.colab.patches import cv2_imshow
import xml.etree.ElementTree as ET

roi_train_dir =  root_dir / "train"

img_files = sorted(roi_train_dir.glob("*.png"))
labels = []

for img in img_files:
    xml_path = annotation_dir / f"{img.stem}.xml"

    tree = ET.parse(xml_path)
    root = tree.getroot()

    label = int(root.find("object").find("name").text.strip())
    labels.append(label)

In [ ]:
print(list(roi_dir.glob("*"))[:10])

In [ ]:
print(len(imgs))
print(imgs[:5])

In [ ]:
print(roi_dir)
print(roi_dir.exists())
print(len(list(roi_dir.iterdir())))
print(list(roi_dir.iterdir())[:5])